In [0]:
from pyspark.sql.functions import (
    col, unix_timestamp, hour, dayofweek,
    month, when
)

df = spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/bronze_2019"
)

print("Initial Row Count:", df.count())

df = df.drop("airport_fee")

df = df.filter(
    (col("fare_amount") > 0) &
    (col("trip_distance") > 0) &
    (col("passenger_count") > 0) &
    col("tpep_pickup_datetime").isNotNull() &
    col("tpep_dropoff_datetime").isNotNull()
)

df = df.withColumn(
    "trip_duration",
    unix_timestamp("tpep_dropoff_datetime") -
    unix_timestamp("tpep_pickup_datetime")
)

df = df.filter(col("trip_duration") > 0)

df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime"))
df = df.withColumn("pickup_month", month("tpep_pickup_datetime"))
df = df.withColumn("pickup_dayofweek", dayofweek("tpep_pickup_datetime"))

df = df.withColumn(
    "is_weekend",
    when(col("pickup_dayofweek").isin([1,7]), 1).otherwise(0)
)

top_n = 1000
categorical_cols = ["PULocationID", "DOLocationID"]

for c in categorical_cols:
    top_values = df.groupBy(c).count() \
                   .orderBy(col("count").desc()) \
                   .limit(top_n) \
                   .select(c)

    top_list = [r[c] for r in top_values.collect()]

    df = df.withColumn(
        f"{c}_idx",
        when(col(c).isin(top_list), col(c)).otherwise(-1)
    )

df = df.withColumn("payment_type_idx", col("payment_type"))

selected_cols = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID_idx",
    "DOLocationID_idx",
    "payment_type_idx",
    "trip_duration",
    "pickup_hour",
    "pickup_month",
    "is_weekend",
    "fare_amount"
]

df_selected = df.select(selected_cols).dropna()

print("Final Cleaned Row Count:", df_selected.count())

df_selected.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/taxi_data/silver_2019"
)

print("Silver layer saved successfully.")